# Upload the trained models to the Hugging Face Hub

The deployed app will download everything from one public Hub repository, so the Space stays small.

**Before you run:**
1. Create a Hugging Face **Write** token at `https://huggingface.co/settings/tokens` (no dot after `.co`).
2. In Colab, click the **key icon** on the left (Secrets), add a secret named `HF_TOKEN`, paste the token as its value, and switch **Notebook access** on. Never paste the token into a cell.
3. Upload `models/type-tfidf-svm.joblib` from your laptop to the Drive folder `issue-triage` (same folder as `issues_clean.parquet`).
4. Set `HF_USER` in the second code cell to your Hugging Face username.

In [ ]:
!pip -q install huggingface_hub
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
HF_USER = "YOUR_HF_USERNAME"          # <- change this
REPO = f"{HF_USER}/issue-triage-artifacts"
DRIVE = "/content/drive/MyDrive/issue-triage"
ART, RUNS = f"{DRIVE}/artifacts", f"{DRIVE}/runs"
TAGS_RUN = "tags-ep5"                 # the chosen tags model (5 epochs)

## Collect the files into one folder

In [ ]:
import os, shutil, json
import pandas as pd

STAGE = "/content/hub_stage"
shutil.rmtree(STAGE, ignore_errors=True); os.makedirs(STAGE)

shutil.copytree(f"{RUNS}/{TAGS_RUN}/best", f"{STAGE}/tags_distilbert")             # DistilBERT for tags
for f in os.listdir(ART):                                                          # priority model and FAISS indexes
    if f.startswith(("faiss_", "priority_")):
        shutil.copy(f"{ART}/{f}", STAGE)
meta = pd.read_parquet(f"{ART}/corpus_meta.parquet")[["repo", "number", "created_at", "title", "source"]]
meta.to_parquet(f"{STAGE}/corpus_meta_slim.parquet", index=False)                  # titles only, no issue bodies
shutil.copy(f"{DRIVE}/type-tfidf-svm.joblib", f"{STAGE}/type_tfidf_svm.joblib")   # issue-type model

json.dump({"types": ["bug", "docs", "feature", "question"],
           "tags": ["api", "build_ci", "needs_discussion", "needs_info", "performance", "regression", "spam_invalid"],
           "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
           "max_len_tags": 256},
          open(f"{STAGE}/labels.json", "w"), indent=1)

open(f"{STAGE}/README.md", "w").write(f"""---
license: mit
---
# Issue triage artifacts

Models and indexes for the GitHub issue triage project (https://github.com/riya28daxini/github-issue-triage):
- `type_tfidf_svm.joblib`: TF-IDF + LinearSVC for issue type
- `tags_distilbert/`: fine-tuned DistilBERT (multi-label) for tags
- `priority_xgb.json`, `priority_config.json`: XGBoost priority model (an engagement proxy)
- `faiss_*.index`, `faiss_*_rows.npy`, `corpus_meta_slim.parquet`: Sentence-BERT duplicate-search indexes (one per repository)
""")

for root, _, files in os.walk(STAGE):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.relpath(p, STAGE):55s} {os.path.getsize(p) / 1e6:8.1f} MB")

## Upload

In [ ]:
from huggingface_hub import login, HfApi
login(userdata.get("HF_TOKEN"))
api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=STAGE, repo_id=REPO, repo_type="model")
print("done: https://huggingface.co/" + REPO)